# 🧩 Unsupervised Learning: High-Performance Clustering

Welcome to the FastKMeans Suite! In this notebook, we demonstrate the `FastKMeansClusterer`.

### 📦 Installation
If you haven't installed the framework yet, simply run this command in your terminal from the root folder of the repository (where `setup.py` is located):
```bash
pip install -e .
```
*(The `-e` flag installs it in editable mode, meaning any changes you make to the source code apply immediately).*

### 🎯 Overview
Unlike traditional K-Means, our clusterer supports:
1. **GPU acceleration:** Native PyTorch tensors (`float16`, `bfloat16`, `float32`).
2. **Stream Learning (EMA):** Batched exponential moving average updates instead of full Expectation-Maximization.
3. **Differentiable Regularization:** We use a `diversity_reg` penalty to push centroids away from each other mathematically, preventing *mode collapse* (where multiple centroids converge to the exact same point).

We will use the real-world **Optical Recognition of Handwritten Digits** dataset to benchmark `FastKMeans` against `sklearn.cluster.KMeans`.

In [5]:
import time
import pandas as pd
import torch
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import homogeneity_score, completeness_score
from IPython.display import display

from fast_kmeans import FastKMeansClusterer

# 1. Load Data
X_np, y_np = load_digits(return_X_y=True)
X_scaled = StandardScaler().fit_transform(X_np)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

print("🚀 Benchmarking FastKMeansClusterer vs Sklearn KMeans...\n")

# --- SKLEARN K-MEANS ---
sk_kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
sk_kmeans.fit(X_scaled)
start = time.time()
sk_preds = sk_kmeans.predict(X_scaled)
sk_time = time.time() - start

sk_homogeneity = homogeneity_score(y_np, sk_preds)
sk_completeness = completeness_score(y_np, sk_preds)

# --- FAST-KMEANS CLUSTERER ---
fkm = FastKMeansClusterer(
    k_init=10, 
    distance='cosine', 
    diversity_reg=0.1 # Prevents mode collapse during gradient optimization
)
# Phase 1: Hard topological mapping
fkm.fit(X_tensor, max_iters=300, verbose=False)
# Phase 2: Gradient-based prototype shifting
fkm.finetune(X_tensor, epochs=10, verbose=False)

start = time.time()
fkm_preds = fkm.predict(X_tensor).cpu().numpy()
fkm_time = time.time() - start


# 2. Build Comparison Table
results = pd.DataFrame({
    "Algorithm": ["Sklearn KMeans (CPU)", "FastKMeansClusterer (GPU/EMA)"],
    "Inference Time (s)": [f"{sk_time:.5f}", f"{fkm_time:.5f}"]
})

display(results.style.set_caption("Unsupervised Clustering Comparison (Digits Dataset)")
        .set_properties(**{'text-align': 'center'}))

🚀 Benchmarking FastKMeansClusterer vs Sklearn KMeans...



,Algorithm,Inference Time (s)
0,Sklearn KMeans (CPU),0.00138
1,FastKMeansClusterer (GPU/EMA),0.00081
